In [ ]:
import os, sys, json, hashlib, shutil, zipfile, numpy as np
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))

from dfthcsos.verify import verify as CV, load_inputs, evaluate_words, passes_accuracy
from dfthcsos.build_angle_k import build_angle_ks
from starter.task3_circuits.angle_k_kickmix import angle_ks_to_circuits, write_kmx_family
from starter.task2_angles.encode import TREES, encode_inputs
from starter.task1_factorization.solution_io import read_factorization

print('Environment ready')


In [ ]:
# Try different chemistry configurations
# Current best: R10/B129/C46, brot=8, J=5, lambda_eff=47.553
# Try variations: different R, B, brot, J

import itertools

configs = [
    # (R, B, C, brot, J, description)
    (10, 129, 46, 8, 5, 'busyaprime baseline'),
    (10, 129, 46, 7, 5, 'brot7'),
    (10, 129, 46, 9, 5, 'brot9'),
    (8, 129, 46, 8, 5, 'R8'),
    (12, 129, 46, 8, 5, 'R12'),
    (10, 100, 46, 8, 5, 'B100'),
    (10, 150, 46, 8, 5, 'B150'),
    (10, 129, 30, 8, 5, 'C30'),
    (10, 129, 60, 8, 5, 'C60'),
    (10, 129, 46, 8, 6, 'J6'),
    (10, 129, 46, 8, 4, 'J4'),
]

print('Will test', len(configs), 'configurations')


In [ ]:
# Load the busyaprime factorization as a base
bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
fac = dict(np.load(bp_members / 'dfthcsos__submission.npz', allow_pickle=False))
print('Base factorization keys:', list(fac.keys()))
for k in fac:
    print(f'  {k}: shape={fac[k].shape}, dtype={fac[k].dtype}')


In [ ]:
# For each config, try to find a matching factorization
# The busyaprime factorization is R10/B129/C46
# We need to search for factorizations with different R, B, C

# Check if there are other factorization files available
import glob
factor_files = glob.glob('/kaggle/input/**/*.npz', recursive=True)
print('Found', len(factor_files), 'npz files')
for f in factor_files[:20]:
    print(' ', f)
